In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

In [2]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, latent_dim)
        )

    def forward(self, x):
        return self.net(x)

In [3]:
def within_cluster_loss(z, labels):
    loss = 0.
    for c in labels.unique():
        mask = (labels == c)
        z_c = z[mask]
        if z_c.size(0) < 2:
            continue
        dist = torch.cdist(z_c, z_c, p=2) ** 2
        loss += dist.sum() / 2
    return loss / z.size(0)

def centroid_loss(z, labels):
    loss = 0.
    for c in labels.unique():
        mask = (labels == c)
        z_c = z[mask]
        centroid = z_c.mean(dim=0, keepdim=True)
        loss += ((z_c - centroid) ** 2).sum()
    return loss / z.size(0)

def silhouette_loss(z, labels):
    n = z.size(0)
    dist = torch.cdist(z, z, p=2)
    same = (labels.unsqueeze(1) == labels.unsqueeze(0)).float()
    same.fill_diagonal_(0)
    a = (dist * same).sum(dim=1) / (same.sum(dim=1) + 1e-8)
    b = torch.full_like(a, float('inf'))
    for c in labels.unique():
        c_mask = (labels == c).float().unsqueeze(0)
        mean_to_c = (dist * c_mask).sum(dim=1) / (c_mask.sum(dim=1) + 1e-8)
        b = torch.min(b, mean_to_c)
    s = (b - a) / (torch.max(a, b) + 1e-8)
    return -s.mean()

In [4]:
transform = transforms.ToTensor()
train_data = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data  = datasets.MNIST('./data', train=False, transform=transform)

# Перевод в float и добавление индексов
train_images = train_data.data.float() / 255.
train_labels = train_data.targets
test_images  = test_data.data.float() / 255.
test_labels  = test_data.targets

train_set = TensorDataset(train_images.unsqueeze(1), train_labels, torch.arange(len(train_images)))
test_set  = TensorDataset(test_images.unsqueeze(1), test_labels, torch.arange(len(test_images)))

train_loader = DataLoader(train_set, batch_size=2048, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=2048, shuffle=False)

In [5]:
from sklearn.cluster import MiniBatchKMeans

def train_with_loss(loss_fn, name, epochs=10, lr=1e-3):
    encoder = Encoder(latent_dim=10)            # без .to(device)
    optimizer = optim.Adam(encoder.parameters(), lr=lr)

    for epoch in range(epochs):
        encoder.eval()
        all_z = []
        with torch.no_grad():
            for x, _, _ in train_loader:
                all_z.append(encoder(x))        # CPU
        all_z = torch.cat(all_z).numpy()

        kmeans = MiniBatchKMeans(n_clusters=10, batch_size=2048, random_state=epoch, n_init=3)
        pseudo = kmeans.fit_predict(all_z)

        encoder.train()
        for x, _, idx in train_loader:
            lbl = torch.tensor(pseudo[idx])
            z = encoder(x)
            loss = loss_fn(z, lbl)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # --- индикация прогресса ---
        print(f'{name} | эпоха {epoch+1}/{epochs} завершена')
    return encoder

In [6]:
def evaluate_encoder(encoder):
    encoder.eval()
    all_z, all_y = [], []
    with torch.no_grad():
        for x, y, _ in test_loader:
            z = encoder(x)
            all_z.append(z)
            all_y.append(y)
    all_z = torch.cat(all_z).numpy()
    all_y = torch.cat(all_y).numpy()

    kmeans = KMeans(n_clusters=10, n_init=20, random_state=42)
    cluster_labels = kmeans.fit_predict(all_z)

    sil = silhouette_score(all_z, cluster_labels)
    ari = adjusted_rand_score(all_y, cluster_labels)
    nmi = normalized_mutual_info_score(all_y, cluster_labels)
    return {'Silhouette': sil, 'ARI': ari, 'NMI': nmi,
            'embeddings': all_z, 'clusters': cluster_labels, 'true_labels': all_y}

In [ ]:
losses = [within_cluster_loss, centroid_loss, silhouette_loss]
names  = ['WithinCluster', 'Centroid', 'Silhouette']
results = {}

for loss_fn, name in zip(losses, names):
    enc = train_with_loss(loss_fn, name, epochs=10)
    results[name] = evaluate_encoder(enc)

WithinCluster | эпоха 1/10 завершена
WithinCluster | эпоха 2/10 завершена
WithinCluster | эпоха 3/10 завершена


In [ ]:
import pandas as pd
metrics_df = pd.DataFrame({name: [res['Silhouette'], res['ARI'], res['NMI']]
                           for name, res in results.items()},
                          index=['Silhouette', 'ARI', 'NMI'])
display(metrics_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, name in zip(axes, names):
    # ограничим выборку для скорости
    n_show = min(5000, len(results[name]['embeddings']))
    idx = np.random.choice(len(results[name]['embeddings']), n_show, replace=False)
    emb_sub = results[name]['embeddings'][idx]
    cl_sub = results[name]['clusters'][idx]

    tsne = TSNE(n_components=2, random_state=42)
    emb_2d = tsne.fit_transform(emb_sub)
    ax.scatter(emb_2d[:, 0], emb_2d[:, 1], c=cl_sub, cmap='tab10', s=3)
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name in zip(axes, names):
    cm = confusion_matrix(results[name]['true_labels'], results[name]['clusters'])
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues')
    ax.set_title(f'Confusion matrix: {name}')
    ax.set_xlabel('Cluster')
    ax.set_ylabel('True class')
plt.tight_layout()
plt.show()